Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q monai scikit-learn


import library


In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)

from monai.transforms import (
    Compose, EnsureChannelFirst, EnsureType,
    ScaleIntensity, Resize,
    RandFlip, RandRotate90, RandZoom,
    RandGaussianNoise, RandAdjustContrast
)
from monai.data import ImageDataset, DataLoader
from monai.networks.nets import DenseNet121

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
base_dir   = '/content/drive/MyDrive/breast cancer'
train_dir  = os.path.join(base_dir, 'train')
valid_dir  = os.path.join(base_dir, 'valid')
test_dir   = os.path.join(base_dir, 'test')
model_path = '/content/drive/MyDrive/best_densenet121.pth'

构建数据列表


In [ ]:
def build_file_list(data_dir):
    image_files, labels = [], []
    for class_name in ['0', '1']:
        class_dir = os.path.join(data_dir, class_name)
        files = glob.glob(os.path.join(class_dir, '*'))
        image_files.extend(files)
        labels.extend([int(class_name)] * len(files))
    return image_files, labels

train_files, train_labels = build_file_list(train_dir)
valid_files, valid_labels = build_file_list(valid_dir)
test_files,  test_labels  = build_file_list(test_dir)

print(f'Train: {len(train_files)} | Valid: {len(valid_files)} | Test: {len(test_files)}')
print(f'Train class 0: {train_labels.count(0)}, class 1: {train_labels.count(1)}')

数据增强


In [ ]:
IMG_SIZE = 224

train_transforms = Compose([
    EnsureChannelFirst(),
    EnsureType(),
    ScaleIntensity(),
    Resize((IMG_SIZE, IMG_SIZE)),
    RandFlip(prob=0.5, spatial_axis=0),
    RandFlip(prob=0.5, spatial_axis=1),
    RandRotate90(prob=0.5),
    RandZoom(prob=0.3, min_zoom=0.85, max_zoom=1.15),
    RandGaussianNoise(prob=0.2, mean=0.0, std=0.05),
    RandAdjustContrast(prob=0.2, gamma=(0.8, 1.2)),
])

val_transforms = Compose([
    EnsureChannelFirst(),
    EnsureType(),
    ScaleIntensity(),
    Resize((IMG_SIZE, IMG_SIZE)),
])

DataLoader

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 2

train_ds = ImageDataset(image_files=train_files, labels=train_labels, transform=train_transforms)
valid_ds = ImageDataset(image_files=valid_files, labels=valid_labels, transform=val_transforms)
test_ds  = ImageDataset(image_files=test_files,  labels=test_labels,  transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}')

模型定义（改进版 DenseNet）

In [ ]:
class ImprovedDenseNet(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.4):
        super().__init__()
        backbone = DenseNet121(spatial_dims=2, in_channels=3, out_channels=1024)
        self.features = backbone.features

        self.classifier = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate / 2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = ImprovedDenseNet(num_classes=2, dropout_rate=0.4).to(device)

训练配置


In [ ]:
MAX_EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 7

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
)

训练循环（核心）

In [ ]:
train_losses, val_accs, val_losses, lr_history = [], [], [], []
best_val_acc = 0.0
patience_count = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss, steps = 0.0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        steps += 1

    avg_train_loss = epoch_loss / steps
    scheduler.step()

训练可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

测试评估


In [ ]:
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

混淆矩阵

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

预测可视化


In [ ]:
sample_imgs, sample_labels = next(iter(test_loader))